# 设置数据库
psql -h 127.0.0.1 -U postgres

# 创建新数据库和新用户
CREATE USER toolbox_user WITH PASSWORD 'my-password';

  CREATE DATABASE toolbox_db;
  GRANT ALL PRIVILEGES ON DATABASE toolbox_db TO toolbox_user;

  ALTER DATABASE toolbox_db OWNER TO toolbox_user;

# 使用新用户连接到数据库
psql -h 127.0.0.1 -U toolbox_user -d toolbox_db

# 创建表
CREATE TABLE hotels(
  id            INTEGER NOT NULL PRIMARY KEY,
  name          VARCHAR NOT NULL,
  location      VARCHAR NOT NULL,
  price_tier    VARCHAR NOT NULL,
  checkin_date  DATE    NOT NULL,
  checkout_date DATE    NOT NULL,
  booked        BIT     NOT NULL
);

# 将数据插入表中
INSERT INTO hotels(id, name, location, price_tier, checkin_date, checkout_date, booked)
VALUES 
  (1, 'Hilton Basel', 'Basel', 'Luxury', '2024-04-22', '2024-04-20', B'0'),
  (2, 'Marriott Zurich', 'Zurich', 'Upscale', '2024-04-14', '2024-04-21', B'0'),
  (3, 'Hyatt Regency Basel', 'Basel', 'Upper Upscale', '2024-04-02', '2024-04-20', B'0'),
  (4, 'Radisson Blu Lucerne', 'Lucerne', 'Midscale', '2024-04-24', '2024-04-05', B'0'),
  (5, 'Best Western Bern', 'Bern', 'Upper Midscale', '2024-04-23', '2024-04-01', B'0'),
  (6, 'InterContinental Geneva', 'Geneva', 'Luxury', '2024-04-23', '2024-04-28', B'0'),
  (7, 'Sheraton Zurich', 'Zurich', 'Upper Upscale', '2024-04-27', '2024-04-02', B'0'),
  (8, 'Holiday Inn Basel', 'Basel', 'Upper Midscale', '2024-04-24', '2024-04-09', B'0'),
  (9, 'Courtyard Zurich', 'Zurich', 'Upscale', '2024-04-03', '2024-04-13', B'0'),
  (10, 'Comfort Inn Bern', 'Bern', 'Midscale', '2024-04-04', '2024-04-16', B'0');

# 以二进制文件形式下载最新版本的 Toolbox：
set OS="windows/amd64"
curl -O https://storage.googleapis.com/genai-toolbox/v0.9.0/$OS/toolbox

# 使用二进制文件可执行
icacls toolbox.exe /grant %USERNAME%:F

# 在MCP上找到相应的tools.yaml：这是下载相应的工具自动生成的文件，不过需要自己再次编写
sources:
  my-pg-source:
    kind: postgres
    host: 127.0.0.1
    port: 5432
    database: toolbox_db
    user: toolbox_user
    password: my-password
tools:
  search-hotels-by-name:
    kind: postgres-sql
    source: my-pg-source
    description: Search for hotels based on name.
    parameters:
      - name: name
        type: string
        description: The name of the hotel.
    statement: SELECT * FROM hotels WHERE name ILIKE '%' || $1 || '%';
  search-hotels-by-location:
    kind: postgres-sql
    source: my-pg-source
    description: Search for hotels based on location.
    parameters:
      - name: location
        type: string
        description: The location of the hotel.
    statement: SELECT * FROM hotels WHERE location ILIKE '%' || $1 || '%';
  book-hotel:
    kind: postgres-sql
    source: my-pg-source
    description: >-
       Book a hotel by its ID. If the hotel is successfully booked, returns a NULL, raises an error if not.
    parameters:
      - name: hotel_id
        type: string
        description: The ID of the hotel to book.
    statement: UPDATE hotels SET booked = B'1' WHERE id = $1;
  update-hotel:
    kind: postgres-sql
    source: my-pg-source
    description: >-
      Update a hotel's check-in and check-out dates by its ID. Returns a message
      indicating  whether the hotel was successfully updated or not.
    parameters:
      - name: hotel_id
        type: string
        description: The ID of the hotel to update.
      - name: checkin_date
        type: string
        description: The new check-in date of the hotel.
      - name: checkout_date
        type: string
        description: The new check-out date of the hotel.
    statement: >-
      UPDATE hotels SET checkin_date = CAST($2 as date), checkout_date = CAST($3
      as date) WHERE id = $1;
  cancel-hotel:
    kind: postgres-sql
    source: my-pg-source
    description: Cancel a hotel by its ID.
    parameters:
      - name: hotel_id
        type: string
        description: The ID of the hotel to cancel.
    statement: UPDATE hotels SET booked = B'0' WHERE id = $1;
toolsets:
  my-toolset:
    - search-hotels-by-name
    - search-hotels-by-location
    - book-hotel
    - update-hotel
    - cancel-hotel

# 运行Toolbox服务器，指向之前创建的文件:tools.yaml
./toolbox --tools-file "tools.yaml"

# 在新终端中，安装SDK包
pip install toolbox-langchain

# 安装其他必需的依赖项
pip install langgraph langchain-google-vertexai

# 以下部分为创建名为hotel_agent.py的文件

In [2]:
import asyncio
import requests  
from typing import Optional, List

In [3]:
class DeepSeekAPI:
    def __init__(self, api_key: str, model: str = "deepseek-chat"):
        self.api_key = api_key
        self.model = model
        self.api_url = "https://api.deepseek.com/v1/chat/completions"  # DeepSeek 官方 API

    def generate(self, prompt: str, max_tokens: int = 1024) -> str:
 
        print(f"传给 DeepSeek 的完整 Prompt:\n{prompt}\n")  

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "temperature": 0.7  
        }
        try:
            response = requests.post(self.api_url, headers=headers, json=payload)
            response.raise_for_status()  
            result = response.json()
            return result["choices"][0]["message"]["content"]
        except Exception as e:
            print(f"DeepSeek API 调用失败: {e}")
            return "（生成失败，请检查 API Key 和网络）"

In [4]:
prompt_template = """
You're a helpful hotel assistant. You handle hotel searching, booking and cancellations...
Always mention hotel IDs while performing any searches...
Don't ask for confirmations from the user.

User query: {query}
"""

queries = [
    "Find hotels in Basel with Basel in it's name.",
    "Can you book the Hilton Basel for me?",
    "Oh wait, this is too expensive. Please cancel it and book the Hyatt Regency instead.",
    "My check in dates would be from April 10, 2024 to April 19, 2024.",
    "Which is the best hotel?"
]

In [5]:
def run_hotel_assistant():

    deepseek = DeepSeekAPI(api_key="sk-578f63b08e74438692e3ebdb42b49934")  

    for query in queries:
     
        full_prompt = prompt_template.format(query=query)
     
        response = deepseek.generate(full_prompt)
        print(f"用户提问: {query}")
        print(f"AI 回复: {response}\n")


In [6]:
if __name__ == "__main__":

    run_hotel_assistant()

传给 DeepSeek 的完整 Prompt:

You're a helpful hotel assistant. You handle hotel searching, booking and cancellations...
Always mention hotel IDs while performing any searches...
Don't ask for confirmations from the user.

User query: Find hotels in Basel with Basel in it's name.


用户提问: Find hotels in Basel with Basel in it's name.
AI 回复: Here are some hotels in Basel with "Basel" in their name:  

1. **Hotel Basel** (ID: HB001)  
2. **Basel Backpack** (ID: HB002)  
3. **Hyperion Hotel Basel** (ID: HB003)  
4. **Hotel Victoria Basel** (ID: HB004)  
5. **Hotel Märthof Basel** (ID: HB005)  

Let me know if you'd like to book any of these or need more details!

传给 DeepSeek 的完整 Prompt:

You're a helpful hotel assistant. You handle hotel searching, booking and cancellations...
Always mention hotel IDs while performing any searches...
Don't ask for confirmations from the user.

User query: Can you book the Hilton Basel for me?


用户提问: Can you book the Hilton Basel for me?
AI 回复: To book the Hilt